In [ ]:
"""
SV callset matching with Truvari-like algorithm
modified to work with breakpoints
"""

import os

workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2/model"
os.chdir(workdir)
os.getcwd()

import importlib
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from pysam import VariantFile
from scipy.ndimage.measurements import label

import svensemble.vcfparse as vcfp
import svensemble._caller_params as caller_params
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.utils as utils
import svensemble.chunk_match as chunk_match

importlib.reload(caller_params)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(utils)
importlib.reload(chunk_match)

pd.set_option('display.max_columns', None)

preprocessed_savedir = Path('saved/preprocessed/')
matched_savedir = Path('saved/matched/')

insert_seq_source = {
    'gnomad': 'svlen',
    'gridss': 'alt',
    'manta': 'svlen'
}


In [23]:
gridss_bp_df = pd.read_csv(preprocessed_savedir / 'gridss_bps.csv')
gnomad_bp_df = pd.read_csv(preprocessed_savedir / 'gnomad_bps.csv')
manta_diploid_bp_df = pd.read_csv(preprocessed_savedir / 'manta_diploid_bps.csv')
bp_df_dict = {'gridss': gridss_bp_df, 'gnomad': gnomad_bp_df, 'manta': manta_diploid_bp_df}

In [ ]:
matched_bp_df = chunk_match.get_connected_pairwise_matches_df(
    bp_df_dict=bp_df_dict,
    insert_size_threshold=0.7,
    max_dist=10
)

matched_bp_df

In [ ]:
info_event_fields = [f'{caller}_INFO_EVENT' for caller in bp_df_dict.keys()]
(~matched_bp_df[info_event_fields].isna()).value_counts()


In [3]:
# gridss_bp_df = gridss_bp_df.sample(frac=0.2)
# gnomad_bp_df = gnomad_bp_df.sample(frac=0.2)

# matches_df = chunk_match.get_ref_comp_match_df(chunk_match.zip_bp_dfs(bp_df_dict), sources_ref=['gnomad'], sources_comp=['gridss', 'manta'])
# matches_df

In [4]:
zipped_bp_df = chunk_match.zip_bp_dfs(bp_df_dict)
zipped_bp_df.loc[:, 'insert_size'] = chunk_match.get_insert_size(zipped_bp_df)

In [5]:
def get_pairwise_matches(
    zipped_bp_df: pd.DataFrame,
    insert_size_threshold: float = 0.7,
    max_dist: int = 10
) -> list[tuple[tuple[str, str], tuple[str, str]]]:
    """
    Match breakpoints from multiple sources based on position and insert size similarity.

    Args:
        zipped_bp_df: DataFrame containing breakpoint data from multiple sources
        insert_size_threshold: Minimum similarity threshold for insert size (0-1)
        max_dist: Maximum allowed distance between breakpoints

    Returns:
        DataFrame containing match matrix with breakpoint IDs as index and columns
    """
    unique_sources = list(zipped_bp_df['source'].unique())
    zipped_bp_df.loc[:, 'insert_size'] = chunk_match.get_insert_size(zipped_bp_df)

    # Generate all possible source combinations
    source_combinations = [
        f'{source1}_{source2}'
        for source1 in unique_sources
        for source2 in unique_sources
    ]

    # Generate combinations between different sources only
    source_combinations_diff = [
        f'{source1}_{source2}'
        for source1 in unique_sources
        for source2 in unique_sources
        if source1 != source2
    ]

    matched_pairs = []

    for chunk in chunk_match.chunker(zipped_bp_df):
        # Skip chunks with insufficient breakpoints or single source
        if len(chunk) <= 1:
            continue

        chunk_df = pd.DataFrame(chunk)
        if len(set(chunk_df['source'])) == 1:
            continue

        # Create source combination masks
        source_masks = {
            source_pair: np.zeros((len(chunk_df), len(chunk_df)), dtype=bool)
            for source_pair in source_combinations
        }

        # Populate source masks
        for i, source1 in enumerate(chunk_df['source'].values):
            for j, source2 in enumerate(chunk_df['source'].values):
                source_masks[f'{source1}_{source2}'][i, j] = True

        # Calculate matrix for different source breakpoints
        sources_matrix = np.sum(
            [
                mask for pair, mask in source_masks.items()
                if pair in source_combinations_diff
            ],
            axis=0
        ).astype(bool)

        # Calculate distance matrices
        bp1_matrix = chunk_match.dist_matrix_ref(chunk_df, chunk_df, 'bnd1')
        bp2_matrix = chunk_match.dist_matrix_ref(chunk_df, chunk_df, 'bnd2')

        # Create position match matrix
        pos_match_matrix = (bp1_matrix <= max_dist) & (bp2_matrix <= max_dist)

        # # Calculate minimum distance matches per source combination
        # dist_sum_matrix = bp1_matrix + bp2_matrix
        # dist_sum_matrix[np.triu_indices_from(dist_sum_matrix)] = np.inf

        # is_min_dist_matrix = np.zeros((len(chunk_df), len(chunk_df)), dtype=bool)
        # for source_pair in source_combinations_diff:
        #     dist_sum_copy = dist_sum_matrix.copy()
        #     dist_sum_copy[~source_masks[source_pair]] = np.inf

        #     # Find minimum in each row and column
        #     row_mins = np.min(dist_sum_copy, axis=1, keepdims=True)
        #     col_mins = np.min(dist_sum_copy, axis=0, keepdims=True)

        #     # Element is minimum if it equals both row and column minimum
        #     min_dist_mask = (dist_sum_copy == row_mins) & (dist_sum_copy == col_mins)
        #     is_min_dist_matrix |= min_dist_mask

        # Calculate insert size similarity
        insert_matrix = (
            chunk_match.rel_insert_size_matrix_ref(chunk_df, chunk_df)
            > insert_size_threshold
        )

        # Create final match matrix
        match_matrix = (
            pos_match_matrix
            & insert_matrix
            # & is_min_dist_matrix
            & ~np.triu(np.ones(len(chunk_df), dtype=bool))
            & sources_matrix
        )
        # Check if any matches exist in the matrix
        if np.any(match_matrix):
            # Iterate through matrix indices
            for i in range(len(chunk_df)):
                for j in range(len(chunk_df)):
                    # If match found, append source and event info for both breakpoints
                    if match_matrix[i, j]:
                        matched_pairs.append([
                            (chunk_df['source'].iloc[i], chunk_df['INFO_EVENT'].iloc[i]),
                            (chunk_df['source'].iloc[j], chunk_df['INFO_EVENT'].iloc[j])
                        ])

    return matched_pairs

In [ ]:
matched_pairs = match_breakpoints(zipped_bp_df=chunk_match.zip_bp_dfs(bp_df_dict))

matched_pairs

In [7]:
from collections import defaultdict

def merge_bp_pairs_dfs(pairs):
    """
    Merge breakpoint pairs into connected components using depth-first search.

    Args:
        pairs: List of tuples representing connected breakpoint pairs

    Returns:
        List of lists, where each inner list contains the connected breakpoints
        in a component, sorted alphabetically
    """
    # Create adjacency list representation of the graph
    adj = defaultdict(set)
    for a, b in pairs:
        adj[a].add(b)
        adj[b].add(a)

    # Track visited nodes and store components
    seen = set()
    components = []

    # Perform DFS for each unvisited node
    for node in adj:
        if node in seen:
            continue

        # Initialize stack and component for current DFS
        stack = [node]
        comp = []

        # DFS traversal
        while stack:
            v = stack.pop()
            if v in seen:
                continue
            seen.add(v)
            comp.append(v)
            # Add unvisited neighbors to stack
            stack.extend(adj[v] - seen)

        # Add sorted component to results
        components.append(sorted(comp))

    return components

In [ ]:
matched_lists = merge_bp_pairs_dfs(matched_pairs)

matched_lists

pd.Series(matched_lists)[pd.Series([len(matched_list) for matched_list in matched_lists]) == 4].iloc[0]

In [ ]:
# # Finally we create a joined dataframe where each row is a combination of multiple bp entries
# # We only keep a single entry per source

# matched_bp_df_list = []

# # We mark each column name with the name of the source as a prefix
# caller_col_names = {source: bp_df.add_prefix(source + '_').columns for source, bp_df in bp_df_dict.items()}

# # Prior to calculations, create a smaller dict of dfs with the column names prefixed
# # And containing only entries that are in the matched list
# unique_matched_events = set()
# for matched_list in matched_lists:
#     for bp_source, bp_event in matched_list:
#         unique_matched_events.add(bp_event)

# bp_df_dict_prefixes = {source: bp_df[bp_df['INFO_EVENT'].isin(unique_matched_events)].add_prefix(source + '_') for source, bp_df in bp_df_dict.items()}

# match_dict = {}
# for source, bp_df in bp_df_dict_prefixes.items():
#     for col in bp_df.columns:
#         match_dict.update({f'{source}_{col}': []})

# sources = list(bp_df_dict_prefixes.keys())
# # Now we can create the joined dataframe
# for matched_list in matched_lists:
#     used_sources = []
#     unused_sources = sources.copy()
#     for bp_source, bp_event in matched_list:
#         if bp_source not in used_sources:
#             matched_bp_df_list.append(bp_df_dict_prefixes[bp_source][bp_df_dict_prefixes[bp_source][f'{bp_source}_INFO_EVENT'] == bp_event])
#             used_sources.append(bp_source)

# matched_bp_df = pd.concat(matched_bp_df_list, ignore_index=True)

# matched_bp_df

In [9]:
# ------------------------------------------------------------------
# build one row per connected component (matched_list)
# ------------------------------------------------------------------
# We mark each column name with the name of the source as a prefix
caller_col_names = {source: bp_df.add_prefix(source + '_').columns for source, bp_df in bp_df_dict.items()}

# Prior to calculations, create a smaller dict of dfs with the column names prefixed
# And containing only entries that are in the matched list
unique_matched_events = set()
for matched_list in matched_lists:
    for bp_source, bp_event in matched_list:
        unique_matched_events.add(bp_event)

bp_df_dict_prefixes = {source: bp_df[bp_df['INFO_EVENT'].isin(unique_matched_events)].add_prefix(source + '_') for source, bp_df in bp_df_dict.items()}

rows = []

for matched_list in matched_lists:
    per_source = []
    seen = set()

    for bp_source, bp_event in matched_list:
        if bp_source in seen:
            continue

        # take *one* row for that (source, EVENT) pair and drop its index
        df = bp_df_dict_prefixes[bp_source].loc[
            bp_df_dict_prefixes[bp_source][f'{bp_source}_INFO_EVENT'] == bp_event
        ].iloc[[0]].reset_index(drop=True)     # <── reset index here !

        per_source.append(df)
        seen.add(bp_source)

    if per_source:                     # join the caller parts side-by-side
        component_row = pd.concat(per_source, axis=1)  # axis=1 = horizontal
        rows.append(component_row)

# finally stack each component *row* under the previous one
matched_bp_df = pd.concat(rows, ignore_index=True)

def build_matched_bp_df(bp_df_dict: dict[str, pd.DataFrame], matched_lists: list[list[tuple[str, str]]]) -> pd.DataFrame:
    """
    Build a DataFrame where each row represents a connected component of matched breakpoints.

    Args:
        bp_df_dict: Dictionary mapping source names to their breakpoint DataFrames
        matched_lists: List of lists, where each inner list contains tuples of (source, event_id)
                      representing connected breakpoints

    Returns:
        DataFrame where each row contains the matched breakpoint data from different sources
        side by side, with source-specific column prefixes
    """
    # Get unique matched events across all components
    unique_matched_events = {
        bp_event
        for matched_list in matched_lists
        for _, bp_event in matched_list
    }

    # Create prefixed DataFrames containing only matched events
    bp_df_dict_prefixes = {
        source: bp_df[bp_df['INFO_EVENT'].isin(unique_matched_events)].add_prefix(source + '_')
        for source, bp_df in bp_df_dict.items()
    }

    rows = []
    for matched_list in matched_lists:
        per_source = []
        seen = set()

        for bp_source, bp_event in matched_list:
            if bp_source in seen:
                continue

            # Get one row for this source/event pair
            df = bp_df_dict_prefixes[bp_source].loc[
                bp_df_dict_prefixes[bp_source][f'{bp_source}_INFO_EVENT'] == bp_event
            ].iloc[[0]].reset_index(drop=True)

            per_source.append(df)
            seen.add(bp_source)

        if per_source:
            # Join source-specific data side by side
            component_row = pd.concat(per_source, axis=1)
            rows.append(component_row)

    # Stack all component rows vertically
    return pd.concat(rows, ignore_index=True)

In [ ]:
caller_col_names

In [ ]:
matched_bp_df = chunk_match.get_connected_pairwise_matches_df(
    bp_df_dict=bp_df_dict,
    insert_size_threshold=0.7,
    max_dist=10
)

matched_bp_df

In [ ]:
info_event_fields = [f'{caller}_INFO_EVENT' for caller in bp_df_dict.keys()]
(~matched_bp_df[info_event_fields].isna()).value_counts()

